# SOXX 评价段原始 N-PORT 普通股集中度

按 `plan.json` 固定 15 份已身份核验的 SEC 原始 XML：逐股 pctVal/netAssets 闭合、30 股内部前五占比与 HHI、2025-09 报告迟到修订差异，原生 Qlib 事件 DatasetH/Recorder。0 收益、拟合和账户。

In [1]:
import os,json,hashlib,decimal,pickle,tarfile,tempfile,subprocess,shlex,xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_soxx_2023_2026_nport_concentration_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
assert sha(root/plan['parent_source_identity'])==plan['parent_source_identity_sha256'] and sha(root/plan['parent_source_transport'])==plan['parent_source_transport_sha256']
source=json.loads((root/plan['parent_source_identity']).read_text());transport=json.loads((root/plan['parent_source_transport']).read_text())
assert source['status']=='fifteen_original_held_NPORT_index_XML_identities_qualified_values_pending' and transport['nas_archive_sha256']==plan['source_NAS_archive_sha256'] and transport['nas_remote_and_scoped_restore_verified']
state={'status':'started','original_XML_reads':0,'new_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
Dnum=decimal.Decimal;local=lambda t:t.rsplit('}',1)[-1]
reports=[];members=[]
for item in source['rows']:
 p=root/plan['raw_root']/(item['accession']+'.XML.xml');assert sha(p)==item['original_XML_sha256'];state['original_XML_reads']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 xml=ET.fromstring(p.read_bytes())
 def values(tag):return sorted({(e.text or '').strip() for e in xml.iter() if local(e.tag)==tag and (e.text or '').strip()})
 assert values('cik')==['0001100663'] and values('seriesId')==['S000004354'] and values('classId')==['C000012084'] and values('repPdDate')==[item['report_period']]
 net_assets=Dnum(values('netAssets')[0]);assert net_assets>0
 holdings=[e for e in xml.iter() if local(e.tag)=='invstOrSec']
 def field(e,name):return next(((q.text or '').strip() for q in e if local(q.tag)==name),'')
 categories={x:sum(field(e,'assetCat')==x for e in holdings) for x in ('EC','DE','STIV')}
 assert categories==item['asset_categories'] and sum(categories.values())==len(holdings) and categories['EC']==30
 equity=[];names=set();cusips=set();max_pct_error=Dnum(0)
 for e in holdings:
  if field(e,'assetCat')!='EC':continue
  assert field(e,'payoffProfile')=='Long' and field(e,'units')=='NS' and field(e,'curCd')=='USD' and field(e,'issuerCat')=='CORP'
  name=field(e,'name');cusip=field(e,'cusip');assert name and name.casefold() not in names;names.add(name.casefold())
  if cusip in {'000000000','N/A'}:identity='placeholder_CUSIP_name_only'
  else:assert cusip and cusip not in cusips;cusips.add(cusip);identity='original_unique_CUSIP'
  balance=Dnum(field(e,'balance'));value=Dnum(field(e,'valUSD'));pct=Dnum(field(e,'pctVal'))
  assert balance>0 and value>0 and pct>0
  error=abs(pct-Dnum(100)*value/net_assets);max_pct_error=max(max_pct_error,error);assert error<=Dnum('0.000001')
  equity.append({'accession':item['accession'],'report_period':item['report_period'],'form':item['form'],'cusip_original':cusip,'identifier_status':identity,'security_original':name,'shares_original':str(balance),'fair_value_usd_original':str(value),'pct_net_assets_original':str(pct),'source_XML_sha256':item['original_XML_sha256']})
 assert len(equity)==30
 total=sum((Dnum(x['fair_value_usd_original']) for x in equity),Dnum(0));weights=[Dnum(x['fair_value_usd_original'])/total for x in equity]
 assert abs(sum(weights)-Dnum(1))<Dnum('1e-25')
 reports.append({'accession':item['accession'],'form':item['form'],'report_period':item['report_period'],'filed':item['filed'],'accepted_display':item['accepted_display'],'net_assets_usd_original':str(net_assets),'EC_value_usd_original':str(total),'EC_rows':30,'other_asset_category_counts':{'DE':categories['DE'],'STIV':categories['STIV']},'max_pct_net_assets_closure_error_pp':str(max_pct_error),'top5_disclosed_EC_value_share':float(sum(sorted(weights,reverse=True)[:5])),'HHI_disclosed_EC_value':float(sum(w*w for w in weights)),'source_XML_sha256':item['original_XML_sha256']})
 members.extend(equity)
assert len(reports)==15 and len(members)==450 and state['original_XML_reads']==15
original=next(x for x in reports if x['report_period']=='2025-09-30' and x['form']=='NPORT-P');amended=next(x for x in reports if x['report_period']=='2025-09-30' and x['form']=='NPORT-P/A')
base_rows={x['security_original']:x for x in members if x['accession']==original['accession']};amend_rows={x['security_original']:x for x in members if x['accession']==amended['accession']}
assert len(base_rows)==len(amend_rows)==30
changes=[]
for name in sorted(set(base_rows)|set(amend_rows)):
 x=base_rows.get(name);y=amend_rows.get(name)
 if x is None or y is None or any(x[k]!=y[k] for k in ('cusip_original','shares_original','fair_value_usd_original')):changes.append({'name':name,'original':x,'amended':y})
amendment={'status':'same_30_original_EC_rows' if not changes else 'amended_EC_rows_differ','original_accession':original['accession'],'amended_accession':amended['accession'],'original_filed':original['filed'],'amended_filed':amended['filed'],'report_period':'2025-09-30','original_EC_value_usd':original['EC_value_usd_original'],'amended_EC_value_usd':amended['EC_value_usd_original'],'top5_original':original['top5_disclosed_EC_value_share'],'top5_amended':amended['top5_disclosed_EC_value_share'],'HHI_original':original['HHI_disclosed_EC_value'],'HHI_amended':amended['HHI_disclosed_EC_value'],'changed_member_rows':len(changes),'changes':changes}
(card/'amendment_comparison.json').write_text(json.dumps(amendment,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
calendar=pd.DatetimeIndex(D.calendar(start_time='2023-01-03',end_time='2026-10-10',freq='day'))
events=pd.DataFrame(reports).sort_values('filed').reset_index(drop=True);events['filed_date']=pd.to_datetime(events.filed)
events['effective_signal_date']=[calendar[int(calendar.searchsorted(d,side='right'))] for d in events.filed_date]
assert events.accession.is_unique and events.effective_signal_date.is_unique and (events.effective_signal_date>events.filed_date).all()
frame=events.set_index('effective_signal_date')[['top5_disclosed_EC_value_share','HHI_disclosed_EC_value']]
frame.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
static=pd.concat({'feature':frame},axis=1);dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'held_events':(str(events.effective_signal_date.min().date()),str(events.effective_signal_date.max().date()))})
assert len(dataset.prepare('held_events',col_set='feature'))==15
source_audit={'status':'held_original_NPORT_15_events_450_EC_rows_source_qualified_no_daily_active_map','source_plan_sha256':sha(card/'plan.json'),'report_events':15,'original_base_events':14,'later_amendment_events':1,'member_rows':450,'placeholder_CUSIP_rows':sum(x['identifier_status']=='placeholder_CUSIP_name_only' for x in members),'original_XML_reads':15,'amendment_status':amendment['status'],'amendment_changed_member_rows':amendment['changed_member_rows'],'first_event_effective_signal':str(events.effective_signal_date.min().date()),'last_event_effective_signal':str(events.effective_signal_date.max().date()),'source_NAS_archive_sha256':plan['source_NAS_archive_sha256'],'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'source_audit.json').write_text(json.dumps(source_audit,indent=2)+'\n')
dataset.config(dump_all=True,recursive=True);assert len(pickle.loads(pickle.dumps(dataset)).prepare('held_events',col_set='feature'))==15
tracking=card/'qlib_recorder'
with R.start(experiment_name='SOXX_held_original_NPORT_concentration_source',uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_hashes={x['accession']:x['source_XML_sha256'] for x in reports})
 rec.save_objects(artifact_path='analysis',source_events=events,member_rows=members,amendment_comparison=amendment,source_audit=source_audit)
 rec.save_objects(artifact_path='metadata',dataset=dataset)
 rec.log_metrics(source_report_events=15,original_EC_rows=450,amendment_changed_member_rows=amendment['changed_member_rows'],model_fits=0,native_accounts=0)
 recorder_id=rec.id;experiment_id=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id);assert len(checked.load_object('analysis/source_events'))==15 and len(checked.load_object('metadata/dataset').prepare('held_events',col_set='feature'))==15
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':'SOXX_held_original_NPORT_concentration_source','ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
with tempfile.TemporaryDirectory(prefix='soxx-held-concentration-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/metadata/dataset') for name in names)
summary={'experiment':card.name,'decision':'held_original_NPORT_equity_concentration_source_qualified_daily_active_map_pending','objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','amendment_comparison':'amendment_comparison.json','report_events':15,'member_rows':450,'model_fits':0,'native_accounts':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','metadata/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['status']='completed';state['recorder_id']=recorder_id;state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({k:source_audit[k] for k in ('report_events','member_rows','placeholder_CUSIP_rows','amendment_status','amendment_changed_member_rows','first_event_effective_signal','last_event_effective_signal')});print({'recorder_id':recorder_id,'nas_sha256':evidence['sha256']})


{'report_events': 15, 'member_rows': 450, 'placeholder_CUSIP_rows': 38, 'amendment_status': 'same_30_original_EC_rows', 'amendment_changed_member_rows': 0, 'first_event_effective_signal': '2023-02-27', 'last_event_effective_signal': '2026-07-14'}
{'recorder_id': '1cb553668fed4767a50a90c2b76fe3d0', 'nas_sha256': 'c59f0c8b1cf45cdc7021765e493f7d543dbffc8cf4c2d63d5d96962b1b1d6574'}


## 中文结论

15 份评价段 SEC 原始 N-PORT 各有 30 条普通股，共 450 条，原始 pctVal 与 valUSD/netAssets 逐行闭合；38 条 CUSIP 占位保留。前五股占比 0.329802–0.383932，HHI 0.046813–0.051563。2025-09 报告于 2026-07 的迟到修订与首版 30 条原值完全相同。Qlib 事件 DatasetH/Recorder `1cb553668fed4767a50a90c2b76fe3d0` 及 NAS SHA-256 `c59f0c8b1cf45cdc7021765e493f7d543dbffc8cf4c2d63d5d96962b1b1d6574` 读回通过。0 收益读取、0 拟合、0 账户；评价段逐日最新报告状态尚待映射。